# Telco Customer Churn — STEP 02. 탐색적 데이터 분석 (EDA)

**목표:** 결측치 비율, 이상치, 변수 분포, 상관관계, 타겟 분포를 확인하고 전처리 실험의 근거를 확보한다.

- 시각화: Histogram, Boxplot, Heatmap, Countplot(Barplot)
- 본 단계는 **탐색 전용**이다. 이상치 삭제나 결측치 대치 등 모델 학습에 영향을 주는 처리는 STEP 03의 학습 데이터 내부 파이프라인에서 비교한다.
- 실행 환경: STEP 01과 동일한 프로젝트 폴더. 로컬 `data/WA_Fn-UseC_-Telco-Customer-Churn.csv`를 우선 사용한다.

## 1. 패키지 및 데이터 불러오기

In [ ]:
# 분석에 필요한 패키지 설치 명령
# %pip install pandas numpy matplotlib seaborn scikit-learn

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', None)

local_candidates = [
    Path('data/WA_Fn-UseC_-Telco-Customer-Churn.csv'),
    Path('WA_Fn-UseC_-Telco-Customer-Churn.csv'),
]
local_file = next((p for p in local_candidates if p.exists()), None)
if local_file is None:
    url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
    df = pd.read_csv(url, dtype={'TotalCharges': 'string'})
    print('Source: IBM public sample URL')
else:
    df = pd.read_csv(local_file, dtype={'TotalCharges': 'string'})
    print('Source:', local_file)

required = {'Churn', 'TotalCharges', 'tenure', 'MonthlyCharges', 'Contract', 'customerID'}
assert required.issubset(df.columns), f'Missing required columns: {required - set(df.columns)}'
print('shape:', df.shape)
display(df.head())

## 2. 데이터 품질 — 결측치 비율 및 중복

`TotalCharges`의 빈 문자열은 일반적인 `isna()` 검사에서 누락될 수 있으므로 수치형으로 변환한 **EDA용 복사본**에서 확인한다.

In [ ]:
eda = df.copy()
eda['TotalCharges'] = pd.to_numeric(eda['TotalCharges'].str.strip(), errors='coerce')
eda['ChurnBinary'] = eda['Churn'].map({'No':0, 'Yes':1})
assert eda['ChurnBinary'].notna().all(), 'Unexpected Churn values'

quality = pd.DataFrame({
    'missing_count': eda.drop(columns='ChurnBinary').isna().sum(),
    'missing_pct': eda.drop(columns='ChurnBinary').isna().mean().mul(100).round(3),
    'dtype': eda.drop(columns='ChurnBinary').dtypes.astype(str),
}).sort_values('missing_pct', ascending=False)
display(quality)
print('전체 중복 행:', df.duplicated().sum())
print('중복 customerID:', df['customerID'].duplicated().sum())
print('TotalCharges 변환 후 결측:', eda['TotalCharges'].isna().sum())

fig, ax = plt.subplots(figsize=(9, 4))
quality.loc[quality['missing_count'] > 0, 'missing_pct'].sort_values().plot.barh(ax=ax)
ax.set(xlabel='Missing values (%)', title='Missing-value ratio (nonzero columns)')
plt.tight_layout()
plt.show()

**데이터 품질:** 수치 변환 후 결측치의 개수와 비율을 집계하고, 문자열 공백과 일반적인 NA를 구분한다.

## 3. 타겟 변수 분포와 클래스 불균형 — Countplot

In [ ]:
counts = eda['Churn'].value_counts().reindex(['No','Yes'])
share = eda['Churn'].value_counts(normalize=True).reindex(['No','Yes']).mul(100)
display(pd.DataFrame({'count':counts,'percent':share.round(2)}))

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=eda, x='Churn', order=['No', 'Yes'], ax=ax)
for i, (count, pct) in enumerate(zip(counts, share)):
    ax.text(i, count + max(counts)*0.01, f'{count:,} ({pct:.1f}%)', ha='center')
ax.set(title='Target distribution: Churn', xlabel='Churn', ylabel='Customers', ylim=(0, max(counts)*1.16))
plt.tight_layout()
plt.show()
print('다수 클래스만 예측했을 때의 정확도:', f'{counts.max()/counts.sum():.3f}')

**클래스 분포:** 유지·이탈 비율을 기준으로 불균형을 확인한다. Accuracy와 Recall, F1, ROC-AUC를 함께 평가한다.

## 4. 수치형 변수 분포 — Histogram

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.histplot(data=eda, x=col, bins=30, ax=ax)
    ax.set_title(col)
fig.suptitle('Numeric feature distributions', y=1.03)
plt.tight_layout()
plt.show()
display(eda[numeric_cols].describe().T)

**분포 분석:** 가입 기간과 월 요금의 집중 구간, 누적 요금의 비대칭성을 히스토그램과 기초 통계량으로 확인한다.

## 5. 이상치 탐색 — Boxplot + IQR 기준

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(data=eda, y=col, ax=ax)
    ax.set_title(col)
fig.suptitle('Numeric boxplots', y=1.03)
plt.tight_layout()
plt.show()

outlier_rows = []
for col in numeric_cols:
    s = eda[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3-q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    count = int(((s < lower) | (s > upper)).sum())
    outlier_rows.append({'feature':col, 'lower_IQR':lower, 'upper_IQR':upper,
                         'outlier_count':count, 'outlier_pct':round(count/len(s)*100,2)})
display(pd.DataFrame(outlier_rows))

**IQR 기준의 해석:** 경계 밖의 값은 이상치 후보이며 기록 오류를 의미하지는 않는다. 실제 요금과 가입 기간에 부합하는지 확인한 후 처리 여부를 판단한다.

## 6. 수치형 상관관계 분석 — Heatmap

In [ ]:
corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'ChurnBinary']
corr = eda[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Pearson correlation (numeric features + target)')
plt.tight_layout()
plt.show()
print('타겟과의 상관계수:')
display(corr['ChurnBinary'].drop('ChurnBinary').sort_values(key=abs, ascending=False).to_frame('correlation'))

**상관관계:** Pearson 계수는 수치형 특성의 선형적 연관성을 나타낸다. 범주형 변수의 중요도나 인과관계를 직접 설명하지 않는다.

## 7. 범주형 변수 특징과 이탈률 — Barplot

In [ ]:
categorical_cols = ['Contract', 'InternetService', 'PaymentMethod']
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, col in zip(axes, categorical_cols):
    rate = eda.groupby(col, dropna=False)['ChurnBinary'].mean().mul(100).sort_values(ascending=False)
    sns.barplot(x=rate.index, y=rate.values, ax=ax)
    ax.set(title=f'Churn rate by {col}', ylabel='Churn rate (%)', xlabel=col, ylim=(0, 100))
    ax.tick_params(axis='x', labelrotation=35)
plt.tight_layout()
plt.show()

for col in categorical_cols:
    print(f'[{col}] 그룹별 고객 수 및 이탈률')
    display(eda.groupby(col, dropna=False)['ChurnBinary'].agg(n='size', churn_rate='mean').assign(churn_rate=lambda t: (t.churn_rate*100).round(2)))

**범주별 분석:** 계약·인터넷 서비스·결제 방식별 고객 수와 이탈률을 함께 비교한다. 표본 규모가 작은 범주의 해석에는 주의가 필요하다.

## 8. EDA 결과 요약용 자동 출력

In [ ]:
churn_pct = eda['ChurnBinary'].mean()*100
missing_col = quality.index[0]
print(f'1) 원본 데이터: {df.shape[0]:,}행, {df.shape[1]}열')
print(f'2) 결측치가 가장 많은 컬럼: {missing_col} ({quality.iloc[0]["missing_count"]}건, {quality.iloc[0]["missing_pct"]:.3f}%)')
print(f'3) 고객 이탈률: {churn_pct:.2f}%')
print(f'4) IQR 기준 이상치 탐색 완료: {", ".join(numeric_cols)}')
print('5) 주요 수치형 상관관계와 범주형 변수별 이탈률 비교 완료')
print('\n결과 요약: 데이터 품질, 클래스 분포 및 주요 변수 관계 확인 완료.')

## 9. 분석 항목 요약

- 공백 문자열을 포함한 결측치 개수와 비율
- IQR 이상치 분석 및 Boxplot
- 수치형 Histogram과 상관관계 Heatmap
- 타겟 Countplot과 클래스 비율
- 범주별 고객 수와 이탈률

전처리 비교는 학습 데이터 내부에서 결측치 대체, 인코딩, 스케일링을 적합한다.